# EPIC Clarity Person Hydration

This notebook hydrates the OMOP PERSON table from EPIC Clarity patient master data.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_PATIENT` - Patient demographics
- `_exponent._bronze_epic_clarity_*.dbo_PATIENT_4` - Extended patient attributes (sex at birth, gender identity)
- `_exponent._bronze_epic_clarity_*.dbo_PATIENT_RACE` - Patient race information

## OMOP Fields Populated
- person_id (surrogate key)
- birth_datetime
- gender_concept_id (from gender_source_value)
- race_concept_id (from race_source_value)
- ethnicity_concept_id (from ethnicity_source_value)
- provider_id (current primary care provider)
- care_site_id (primary care location)
- location_id (residential location)

In [ ]:
source = 'epic_clarity'

In [ ]:
-- Silver Layer: Transform EPIC patient data
%sql
CREATE OR REPLACE TEMP VIEW person_silver AS
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', p.PAT_ID) AS person_source_value,
    p.BIRTH_DATE AS birth_datetime,
    YEAR(p.BIRTH_DATE) AS year_of_birth,
    MONTH(p.BIRTH_DATE) AS month_of_birth,
    DAY(p.BIRTH_DATE) AS day_of_birth,
    COALESCE(p4.SEX_ASGN_AT_BIRTH_C_NAME, '') AS gender_source_value,
    COALESCE(pr.PATIENT_RACE_C_NAME, '') AS race_source_value,
    COALESCE(p.ETHNIC_GROUP_C_NAME, '') AS ethnicity_source_value,
    CONCAT_WS(CHR(31), 'epic_clarity', 'CLARITY_SER', 'PROV_ID', p.CUR_PCP_PROV_ID_PROV_NAME) AS provider_source_value,
    CONCAT_WS(CHR(31), 'epic_clarity', 'CLARITY_DEP', 'DEPARTMENT_ID', p.CUR_PRIM_LOC_ID_LOC_NAME) AS care_site_source_value,
    CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', p.PAT_ID) AS location_source_value,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity_prod01_vw.dbo_PATIENT p
LEFT JOIN _exponent._bronze_epic_clarity_prod01_vw.dbo_PATIENT_4 p4
    ON p.PAT_ID = p4.PAT_ID
LEFT JOIN _exponent._bronze_epic_clarity_prod01_vw.dbo_PATIENT_RACE pr
    ON p.PAT_ID = pr.PAT_ID
WHERE p.PAT_ID IS NOT NULL

In [ ]:
-- Merge into Silver Layer
%sql
MERGE INTO _exponent.omop_silver.person AS target
USING person_silver AS source
ON target.person_source_value = source.person_source_value

WHEN MATCHED AND NOT (
    target.birth_datetime <=> source.birth_datetime
    AND target.gender_source_value <=> source.gender_source_value
    AND target.race_source_value <=> source.race_source_value
    AND target.ethnicity_source_value <=> source.ethnicity_source_value
    AND target.provider_source_value <=> source.provider_source_value
    AND target.care_site_source_value <=> source.care_site_source_value
    AND target.location_source_value <=> source.location_source_value
)
THEN UPDATE SET
    target.birth_datetime = source.birth_datetime,
    target.year_of_birth = source.year_of_birth,
    target.month_of_birth = source.month_of_birth,
    target.day_of_birth = source.day_of_birth,
    target.gender_source_value = source.gender_source_value,
    target.race_source_value = source.race_source_value,
    target.ethnicity_source_value = source.ethnicity_source_value,
    target.provider_source_value = source.provider_source_value,
    target.care_site_source_value = source.care_site_source_value,
    target.location_source_value = source.location_source_value,
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    person_source_value,
    birth_datetime,
    year_of_birth,
    month_of_birth,
    day_of_birth,
    gender_source_value,
    race_source_value,
    ethnicity_source_value,
    provider_source_value,
    care_site_source_value,
    location_source_value,
    updated_tsp
)
VALUES (
    source.person_source_value,
    source.birth_datetime,
    source.year_of_birth,
    source.month_of_birth,
    source.day_of_birth,
    source.gender_source_value,
    source.race_source_value,
    source.ethnicity_source_value,
    source.provider_source_value,
    source.care_site_source_value,
    source.location_source_value,
    source.updated_tsp
)

In [ ]:
-- Populate mapping table
%sql
INSERT INTO _exponent.omop_mapping.source_to_person (
    source_system,
    person_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    'epic_clarity' AS source_system,
    s.person_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    s.updated_tsp AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT person_source_value, updated_tsp
    FROM _exponent.omop_silver.person
    WHERE person_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_person x
    ON s.person_source_value = x.person_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
-- Gold Layer: Join with mapping and concept tables to get person_id and concept IDs
%sql
CREATE OR REPLACE TEMP VIEW person_gold AS
SELECT
    m.person_id,
    s.birth_datetime,
    s.year_of_birth,
    s.month_of_birth,
    s.day_of_birth,
    COALESCE(gc.concept_id, 0) AS gender_concept_id,
    COALESCE(rc.concept_id, 0) AS race_concept_id,
    COALESCE(ec.concept_id, 0) AS ethnicity_concept_id,
    mp.provider_id,
    mc.care_site_id,
    ml.location_id,
    s.updated_tsp
FROM _exponent.omop_silver.person s
INNER JOIN _exponent.omop_mapping.source_to_person m
    ON s.person_source_value = m.person_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept gc
    ON gc.source_id = s.gender_source_value
    AND gc.domain_id = 'Gender'
    AND gc.source_system = 'epic_clarity'
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept rc
    ON rc.source_id = s.race_source_value
    AND rc.domain_id = 'Race'
    AND rc.source_system = 'epic_clarity'
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept ec
    ON ec.source_id = s.ethnicity_source_value
    AND ec.domain_id = 'Ethnicity'
    AND ec.source_system = 'epic_clarity'
LEFT JOIN _exponent.omop_mapping.source_to_provider mp
    ON s.provider_source_value = mp.provider_source_value
    AND mp.source_system = 'epic_clarity'
    AND mp.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_care_site mc
    ON s.care_site_source_value = mc.care_site_source_value
    AND mc.source_system = 'epic_clarity'
    AND mc.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_location ml
    ON s.location_source_value = ml.location_source_value
    AND ml.source_system = 'epic_clarity'
    AND ml.active_flag = TRUE

In [ ]:
-- Merge into Gold Layer (OMOP)
%sql
MERGE INTO _exponent.omop.person AS target
USING person_gold AS source
ON target.person_id = source.person_id

WHEN MATCHED AND NOT (
    target.birth_datetime <=> source.birth_datetime
    AND target.gender_concept_id <=> source.gender_concept_id
    AND target.race_concept_id <=> source.race_concept_id
    AND target.ethnicity_concept_id <=> source.ethnicity_concept_id
    AND target.provider_id <=> source.provider_id
    AND target.care_site_id <=> source.care_site_id
    AND target.location_id <=> source.location_id
)
THEN UPDATE SET
    target.birth_datetime = source.birth_datetime,
    target.year_of_birth = source.year_of_birth,
    target.month_of_birth = source.month_of_birth,
    target.day_of_birth = source.day_of_birth,
    target.gender_concept_id = source.gender_concept_id,
    target.race_concept_id = source.race_concept_id,
    target.ethnicity_concept_id = source.ethnicity_concept_id,
    target.provider_id = source.provider_id,
    target.care_site_id = source.care_site_id,
    target.location_id = source.location_id

WHEN NOT MATCHED THEN INSERT (
    person_id,
    birth_datetime,
    year_of_birth,
    month_of_birth,
    day_of_birth,
    gender_concept_id,
    race_concept_id,
    ethnicity_concept_id,
    provider_id,
    care_site_id,
    location_id
)
VALUES (
    source.person_id,
    source.birth_datetime,
    source.year_of_birth,
    source.month_of_birth,
    source.day_of_birth,
    source.gender_concept_id,
    source.race_concept_id,
    source.ethnicity_concept_id,
    source.provider_id,
    source.care_site_id,
    source.location_id
)